# Visuelle Oberkategorien: Beispielabbildungen

Erzeugt Beispielabbildungen zu struktureller Dynamik, Bildqualität, Ästhetik/Optimierung sowie Emotion/Präsenz.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
__file__ = str(PROJECT_ROOT / "exports" / "notebooks" / "06_visuelle_oberkategorien_beispielabbildungen_notebook.py")
print(f"Projektordner: {PROJECT_ROOT}")


## Code


In [ ]:
from __future__ import annotations

import csv
import shutil
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont, ImageOps


ROOT = Path(__file__).resolve().parents[2]
OUT_DIR = ROOT / "exports" / "examples" / "figures"
FRAME_DIR = ROOT / "exports" / "examples" / "frames"
METADATA_CSV = ROOT / "exports" / "examples" / "metadata" / "screenshot_examples_metadata.csv"

TILE_W = 420
TILE_H = 746
LABEL_H = 170
GAP = 26
MARGIN = 40

BG = (248, 247, 242)
PANEL_BG = (255, 255, 255)
TEXT = (29, 32, 36)
MUTED = (76, 83, 92)
VI = (105, 73, 160)
RI = (32, 118, 117)


def font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf"),
        Path("/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        Path("C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size)
    return ImageFont.load_default()


FONT_TITLE = font(21, True)
FONT_SMALL = font(15)


def read_metadata() -> list[dict[str, str]]:
    with METADATA_CSV.open(encoding="utf-8-sig", newline="") as fh:
        return list(csv.DictReader(fh))


def find_row(rows: list[dict[str, str]], figure: str, position: int) -> dict[str, str]:
    for row in rows:
        if row["figure"] == figure and int(row["position"]) == position:
            return row
    raise ValueError(f"Missing metadata row for {figure} position {position}")


def source_frame(row: dict[str, str]) -> Path:
    stem = Path(row["figure"]).stem
    path = FRAME_DIR / f"{stem}_{int(row['position']):02d}_{row['type']}_{row['video_id']}.jpeg"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def fit_image(path: Path) -> Image.Image:
    image = Image.open(path).convert("RGB")
    return ImageOps.fit(image, (TILE_W, TILE_H), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))


def draw_wrapped(
    draw: ImageDraw.ImageDraw,
    xy: tuple[int, int],
    text: str,
    max_width: int,
    fnt: ImageFont.ImageFont,
    fill=TEXT,
    max_lines: int = 2,
) -> int:
    words = str(text).split()
    lines: list[str] = []
    line = ""
    for word in words:
        candidate = f"{line} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=fnt)[2] <= max_width or not line:
            line = candidate
        else:
            lines.append(line)
            line = word
    if line:
        lines.append(line)

    x, y = xy
    line_height = draw.textbbox((0, 0), "Ag", font=fnt)[3] + 6
    for visible_line in lines[:max_lines]:
        draw.text((x, y), visible_line, font=fnt, fill=fill)
        y += line_height
    return y


def compact_subtitle(text: str, max_lines: int = 3) -> str:
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    kept: list[str] = []
    for line in lines:
        if line.startswith("Modell:"):
            kept.append(line)
        if len(kept) >= max_lines:
            break
    return "\n".join(kept)


def draw_tile(
    canvas: Image.Image,
    draw: ImageDraw.ImageDraw,
    item: dict[str, str],
    x: int,
    y: int,
    figure_stem: str,
    pos: int,
) -> dict[str, str]:
    src = source_frame(item)
    canvas.paste(fit_image(src), (x, y))

    dst = FRAME_DIR / f"{figure_stem}_{pos:02d}_{item['type']}_{item['video_id']}.jpeg"
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

    stripe = VI if item["type"] == "VI" else RI
    draw.rectangle((x, y + TILE_H, x + TILE_W, y + TILE_H + LABEL_H), fill=PANEL_BG)
    draw.rectangle((x, y + TILE_H, x + 9, y + TILE_H + LABEL_H), fill=stripe)

    label_x = x + 20
    label_y = y + TILE_H + 7
    title = f"{item['overview_label']} | {item['type']} | id={str(item['video_id'])[-6:]}"
    label_y = draw_wrapped(draw, (label_x, label_y), title, TILE_W - 32, FONT_TITLE, fill=TEXT, max_lines=1)
    subtitle = compact_subtitle(item["subtitle"])
    for raw_line in subtitle.splitlines():
        label_y = draw_wrapped(draw, (label_x, label_y + 1), raw_line, TILE_W - 32, FONT_SMALL, fill=MUTED, max_lines=2)

    record = {
        "figure": f"{figure_stem}.png",
        "position": str(pos),
        "video_id": item["video_id"],
        "type": item["type"],
        "title": title,
        "subtitle": subtitle,
        "overview_label": item["overview_label"],
        "source_figure": item["figure"],
        "source_position": item["position"],
        "frame_path": str(dst.relative_to(ROOT)),
        "source_frame": str(src.relative_to(ROOT)),
    }
    return record


def render(path: Path, title: str, items: list[dict[str, str]], columns: int) -> list[dict[str, str]]:
    rows = (len(items) + columns - 1) // columns
    width = MARGIN * 2 + columns * TILE_W + (columns - 1) * GAP
    height = MARGIN * 2 + rows * (TILE_H + LABEL_H) + (rows - 1) * GAP
    canvas = Image.new("RGB", (width, height), BG)
    draw = ImageDraw.Draw(canvas)

    records: list[dict[str, str]] = []
    y0 = MARGIN
    for idx, item in enumerate(items, start=1):
        row = (idx - 1) // columns
        col = (idx - 1) % columns
        x = MARGIN + col * (TILE_W + GAP)
        y = y0 + row * (TILE_H + LABEL_H + GAP)
        records.append(draw_tile(canvas, draw, item, x, y, path.stem, idx))

    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path, quality=95)
    return records


def write_metadata(path: Path, records: list[dict[str, str]]) -> None:
    fieldnames = [
        "figure",
        "position",
        "video_id",
        "type",
        "title",
        "subtitle",
        "overview_label",
        "source_figure",
        "source_position",
        "source_frame",
        "frame_path",
    ]
    with path.open("w", encoding="utf-8-sig", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)


def build_items(rows: list[dict[str, str]], specs: list[tuple[str, int, str]]) -> list[dict[str, str]]:
    items = []
    for figure, position, label in specs:
        row = dict(find_row(rows, figure, position))
        row["overview_label"] = label
        items.append(row)
    return items


def main() -> None:
    rows = read_metadata()

    figures = [
        (
            "abb_oberkategorie_strukturelle_inszenierung_visuelle_dynamik.png",
            "Strukturelle Inszenierung und visuelle Dynamik",
            3,
            [
                ("abb_kategorie_cuts_v02.png", 1, "Schnittdynamik"),
                ("abb_kategorie_camera_stability_v03.png", 1, "Kamerastabilität"),
                ("abb_kategorie_camera_distance_v01.png", 2, "Kameradistanz"),
                ("abb_visuelle_settings_szenen_v01.png", 2, "Szenenkontext"),
                ("abb_kategorie_scene_v01.png", 2, "Szenenvielfalt"),
                ("abb_kategorie_personen_anzahl_v05.png", 1, "Personenanzahl"),
            ],
        ),
        (
            "abb_oberkategorie_bildtechnische_qualitaet.png",
            "Bildtechnische Qualität",
            3,
            [
                ("abb_kategorie_brightness_v01.png", 1, "Helligkeit"),
                ("abb_kategorie_contrast_v05.png", 3, "Kontrast"),
                ("abb_kategorie_saturation_v04.png", 3, "Farbsättigung"),
                ("abb_kategorie_sharpness_v05.png", 1, "Schärfe"),
                ("abb_kategorie_brightness_v05.png", 3, "Helligkeit"),
                ("abb_kategorie_sharpness_v02.png", 1, "Schärfe"),
            ],
        ),
        (
            "abb_oberkategorie_aesthetik_visuelle_optimierung.png",
            "Ästhetik und visuelle Optimierung",
            3,
            [
                ("abb_kategorie_aesthetic_quality_v05.png", 1, "Aesthetic Score"),
                ("abb_kategorie_beauty_score_v02.png", 1, "Beauty Score"),
                ("abb_kategorie_skin_smoothness_v02.png", 1, "Hautglättung"),
                ("abb_kategorie_filter_strength_v05.png", 1, "Filterindikator"),
                ("abb_kategorie_aesthetic_quality_v05.png", 2, "Aesthetic Score"),
                ("abb_kategorie_beauty_score_v01.png", 3, "Beauty Score"),
            ],
        ),
        (
            "abb_oberkategorie_emotion_praesenz.png",
            "Emotion und Präsenz",
            3,
            [
                ("abb_kategorie_face_emotion_v02.png", 1, "Gesichtsemotion"),
                ("abb_kategorie_face_orientation_v03.png", 1, "Gesichtsausrichtung"),
                ("abb_kategorie_body_pose_v03.png", 1, "Körperpose"),
                ("abb_kategorie_personen_anzahl_v05.png", 3, "Personenanzahl"),
                ("abb_kategorie_face_emotion_v03.png", 2, "Gesichtsemotion"),
                ("abb_kategorie_face_orientation_v05.png", 1, "Gesichtsausrichtung"),
            ],
        ),
    ]

    for filename, title, columns, specs in figures:
        path = OUT_DIR / filename
        items = build_items(rows, specs)
        records = render(path, title, items, columns)
        write_metadata(OUT_DIR / filename.replace(".png", "_metadata.csv"), records)
        print(f"Created {path}")



## Ausführung


In [ ]:
main()
